# MLA Week 5 Lab — SVM and Regularization
**Machine Learning Applications (MLA), Hanoi University (HANU)** · Semester 1, AY 2026–2027

This lab turns the Week 5 lecture into working code. You will compute a margin
**from first principles**, implement the hinge loss and train a linear SVM by
subgradient descent, then compare L1 and L2 regularization on a synthetic
Eedi-style task and choose the penalty strength honestly with a validation
curve.

> **Data note.** The datasets here are generated inside the notebook, so the lab
> is self-contained and runs on Google Colab with no file upload. The *same
> modeling pattern* is intended to transfer to the real Eedi response CSVs
> (`train_data.csv`, ...) after adapting the data-loading and feature-
> construction cell; that step is not run in this lab.

## 1. Learning objectives

By the end of this lab you should be able to:

1. compute the margin `y(wᵀx + b)` and the corridor width `2/‖w‖`, identify which points lie on the margin, and distinguish those from the support vectors (the points with a non-zero dual coefficient);
2. implement the hinge loss and read off which points contribute nothing;
3. train a linear SVM from scratch by subgradient descent on `hinge + (λ/2)‖w‖²`;
4. show what the `C` dial does to the margin and to the number of support vectors;
5. compare L1 and L2 penalties on the same data, and count how many weights L1 sets to exactly zero;
6. choose `λ` with a validation curve and with k-fold cross-validation, without touching the test set;
7. demonstrate why features must be standardised before they are regularised;
8. state what a sparse feature list does and does not tell you about learning.

## 2. Before you start — 6-point notebook check

Confirm you can do all six **before** the SVM content, so that a notebook
mechanic is not what blocks you later. Full instructions are in
`08_Student_SelfStudy/START_HERE_WEEK5.pdf` §4b.

| # | Can you… | If not |
|---|---|---|
| 1 | Run cells from the top, in order, and see output under each | **Run → Run All Above**, or `Shift+Enter` cell by cell |
| 2 | Read the version numbers printed by the setup cell below | It is the next code cell |
| 3 | Replace one `TODO` (try the first in Part A) and re-run that cell | Delete the `pass` line and write the code the comment describes |
| 4 | Print a NumPy array and say how many numbers it holds | `print(X)` then `print(X.shape)` |
| 5 | Read a failed assertion and name which value was wrong | The message is written to say so — read the text after the comma |
| 6 | Restart the kernel and re-run everything from clean | **Kernel → Restart & Run All** |

## 3. Setup and imports

In [ ]:
# Core scientific stack. numpy for the maths, matplotlib for plots.
import sys
import numpy as np
import matplotlib.pyplot as plt
import sklearn

rng = np.random.default_rng(42)   # fixed seed: results are reproducible
np.set_printoptions(precision=4, suppress=True)

# Environment check. If a version differs from the one this lab was built
# against, the narrative still holds but the last decimal place may not.
print("python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
assert sys.version_info >= (3, 8), "This lab needs Python 3.8 or newer."
assert int(sklearn.__version__.split(".")[0]) >= 1, \
    "This lab needs scikit-learn 1.x (penalty=None requires >= 1.2)."
print("\nBuilt and verified against: python 3.12.7, numpy 2.x, scikit-learn 1.x")

## 4. Part A — The margin, by hand

We start from the exact five points used in the lecture and the tutorial:

| point | $(1,1)$ | $(1,3)$ | $(3,1)$ | $(3,3)$ | $(5,2)$ |
|---|---|---|---|---|---|
| $y$ | $-1$ | $-1$ | $+1$ | $+1$ | $+1$ |

The maximum-margin model is $w = (1,0)$, $b = -2$. Implement the margin
$y(w^\top x + b)$ and the corridor width $2/\|w\|$, then confirm that four
points sit at margin exactly $1$ and one sits at $3$.

In [ ]:
X = np.array([[1., 1.], [1., 3.], [3., 1.], [3., 3.], [5., 2.]])
y = np.array([-1., -1., 1., 1., 1.])
w = np.array([1., 0.])
b = -2.0

def margins(X, y, w, b):
    # TODO: return y * (X @ w + b), one value per point
    pass

def corridor_width(w):
    # TODO: return 2 / ||w||   [np.linalg.norm]
    pass

# TODO: print the scores, the margins, and the width
# TODO: print the points whose margin is exactly 1 (use np.isclose)
#       These are the points ON the margin. Which of them are support vectors
#       is a separate question, settled by the dual coefficients below.

**Expected:** `margins = [1, 1, 1, 1, 3]` and `width = 2.0`. The four students at
margin `1` sit on the margin; Giang `(5,2)` at margin `3` does not.

In [ ]:
# Self-check for Part A. Run it after your TODOs; it tells you WHICH bit is wrong.
_m = margins(X, y, w, b)
assert _m is not None, "margins() returned None - did you replace the `pass`?"
assert np.allclose(_m, [1, 1, 1, 1, 3]), \
    f"margins should be [1,1,1,1,3], got {_m}. Did you multiply by y?"
assert np.isclose(corridor_width(w), 2.0), \
    f"width should be 2.0, got {corridor_width(w)}. It is 2/||w||, not 1/||w||."
print("Part A self-check passed.")

### Cross-check against `scikit-learn`

A hard-margin SVM is approximated by a soft-margin SVM with a very large `C`.
The fitted `w` and `b` should match our hand solution.

In [ ]:
from sklearn.svm import SVC

clf = SVC(kernel="linear", C=1e6).fit(X, y)
print("sklearn w =", clf.coef_.ravel(), "  b =", clf.intercept_[0])
print("our     w =", w, "  b =", b)
print("sklearn margin width =", 2.0 / np.linalg.norm(clf.coef_))

# Support vectors are formally the points with a dual coefficient alpha > 0.
# Every such point lies ON the margin, but the converse can fail: here four
# points sit at margin 1 and they are not independent, so the same boundary can
# be written using only a subset of them. sklearn reports one valid subset.
print("points ON the margin (margin == 1):")
print(X[np.isclose(margins(X, y, clf.coef_.ravel(), clf.intercept_[0]), 1.0)])
print("subset sklearn reports as support_:")
print(clf.support_vectors_)
print("dual coefficients alpha*y :", clf.dual_coef_.ravel())

**Read this carefully — it is the one subtle point in Part A.** Four students
sit exactly on the margin, but `sklearn` reports fewer in `support_`. Both are
right. A support vector is formally a point with a non-zero dual coefficient
$\alpha_i$; every such point lies on the margin, but when several margin points
are redundant the optimiser can reproduce the identical boundary using only some
of them. So:

- *on the margin* = **could** hold the boundary up (what we can check by hand);
- $\alpha_i > 0$ = **does** hold it up in the solution the solver returned.

In a non-degenerate problem the two sets coincide, which is why the lecture
treats them as one idea.

## 5. Part B — Hinge loss

The hinge loss is

$$L_i = \max\big(0,\ 1 - y_i(w^\top x_i + b)\big).$$

Add Mai from the lecture -- $(1, 2)$ with $y = +1$, one practice question
right but she passed anyway -- and confirm her hinge loss is $2$ while the five
original students all score $0$.

In [ ]:
def hinge(X, y, w, b):
    # TODO: return max(0, 1 - margin) elementwise  [np.maximum]
    pass

X_noisy = np.vstack([X, [1.0, 2.0]])   # Mai
y_noisy = np.r_[y, 1.0]

# TODO: compute and print the margins, the hinge losses, and their sum

**Expected:** `hinge = [0, 0, 0, 0, 0, 2.0]`. Five students contribute nothing at
all. Only Mai pulls on the weights — this is why the fitted model depends
on so few points.

In [ ]:
# Self-check for Part B.
_h = hinge(X_noisy, y_noisy, w, b)
assert _h is not None, "hinge() returned None - did you replace the `pass`?"
assert (_h >= 0).all(), "a hinge loss can never be negative - did you forget np.maximum?"
assert np.allclose(_h, [0, 0, 0, 0, 0, 2.0]), \
    f"hinge should be [0,0,0,0,0,2], got {_h}"
print("Part B self-check passed. Mai's hinge loss =", _h[-1])

## 6. Part C — A linear SVM from scratch

We minimise the regularised hinge objective

$$J(w,b) = \frac{1}{N}\sum_i \max\big(0,\ 1 - y_i(w^\top x_i + b)\big)
  + \frac{\lambda}{2}\|w\|^2$$

by subgradient descent. The hinge has a kink, so we use a *subgradient*: points
with margin $< 1$ contribute $-y_i x_i$, and points already safe contribute
nothing. The penalty always contributes $\lambda w$.

In [ ]:
# Two overlapping classes: not separable, so the soft margin is doing real work.
n = 200
Xa = rng.normal([1.6, 2.0], [0.6, 0.9], size=(n, 2))
Xb = rng.normal([3.6, 2.0], [0.6, 0.9], size=(n, 2))
X2 = np.vstack([Xa, Xb])
y2 = np.r_[-np.ones(n), np.ones(n)]

plt.figure(figsize=(5.4, 3.4))
plt.scatter(Xa[:, 0], Xa[:, 1], marker="o", facecolor="white",
            edgecolor="#0D2137", s=22, label="y = -1")
plt.scatter(Xb[:, 0], Xb[:, 1], marker="s", color="#0D2137", s=22, label="y = +1")
plt.xlabel("x1"); plt.ylabel("x2"); plt.legend()
plt.title("Overlapping classes"); plt.tight_layout(); plt.show()

In [ ]:
def train_svm(X, y, lam=0.01, alpha=0.05, epochs=800):
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []
    N = len(y)
    for _ in range(epochs):
        m = y * (X @ w + b)
        # TODO: viol = boolean mask of points with margin < 1
        # TODO: grad_w = -(X[viol].T @ y[viol]) / N + lam * w
        # TODO: grad_b = -y[viol].sum() / N
        # TODO: update w and b by -alpha * gradient
        # TODO: append the objective  mean(max(0, 1-m)) + 0.5*lam*w@w
        pass
    return w, b, history

# w_svm, b_svm, hist = train_svm(X2, y2, lam=0.01, alpha=0.05, epochs=800)
# print("w =", w_svm, "b =", b_svm)
# plt.plot(hist); plt.xlabel("epoch"); plt.ylabel("objective"); plt.show()

The objective should fall quickly and then flatten. The learned `w` points from
the negative class toward the positive one, so its first component is clearly
positive and its second is near zero — the vertical feature carries no
separating information here.

In [ ]:
# Cross-check: sklearn's linear SVM on the same data, with C matched to our lambda.
# Our objective averages the hinge and uses (lam/2)||w||^2, so C = 1 / (lam * N).
from sklearn.svm import LinearSVC

C_equiv = 1.0 / (0.01 * len(y2))
ref = LinearSVC(C=C_equiv, loss="hinge", max_iter=200000).fit(X2, y2)
print("sklearn   w =", ref.coef_.ravel(), " b =", round(ref.intercept_[0], 4))
print("ours      w =", w_svm, " b =", round(b_svm, 4))
print("sklearn accuracy =", round(ref.score(X2, y2), 4))
print("ours     accuracy =", round(np.mean(np.sign(X2 @ w_svm + b_svm) == y2), 4))

## 7. Part D — What the `C` dial actually does

`C` is the price of a violation. Large `C` means violations are expensive, so
the model narrows the corridor to avoid them. Watch the margin width shrink and
the support-vector count fall as `C` rises.

In [ ]:
rows = []
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = SVC(kernel="linear", C=C).fit(X2, y2)
    rows.append((C, 2 / np.linalg.norm(m.coef_), len(m.support_), m.score(X2, y2)))

print(f"{'C':>8} {'margin width':>14} {'#support vec':>14} {'train acc':>11}")
for C, width, nsv, a in rows:
    print(f"{C:>8g} {width:>14.3f} {nsv:>14d} {a:>11.3f}")

Read the table as a sentence: as `C` goes up, the margin narrows and the model
leans on fewer points. That is *less* regularization, not more — the single most
common error of this week.

## 8. Part E — A synthetic Eedi-style dataset

Each row is one attempt: a student meets a question and answers it correctly
(`1`) or not (`0`). Four features carry real signal; sixteen are noise, which is
realistic once you start engineering features from a response log. This is the
setting where regularization earns its place.

In [ ]:
N, D = 400, 20
Xf = rng.normal(size=(N, D))

# Only the first four features actually drive the outcome.
w_true = np.zeros(D)
w_true[:4] = [1.8, -1.5, 1.0, 0.7]     # accuracy, difficulty, topic match, practice
p_true = 1.0 / (1.0 + np.exp(-(Xf @ w_true)))
yf = (rng.uniform(size=N) < p_true).astype(int)

# Train / validation / test split (50 / 25 / 25).
idx = rng.permutation(N)
tr, va, te = idx[:200], idx[200:300], idx[300:]
Xtr, ytr = Xf[tr], yf[tr]
Xva, yva = Xf[va], yf[va]
Xte, yte = Xf[te], yf[te]
print("real features:", 4, " noise features:", D - 4)
print("class balance (fraction correct):", round(yf.mean(), 3))
print("train / val / test sizes:", len(tr), len(va), len(te))

## 9. Part F — L1 against L2

Fit logistic regression three ways: no penalty, L2, and L1. Then count how many
weights each one sets to *exactly* zero.

`scikit-learn` takes `C`, not `λ`, and `C` is inversely related to the penalty
strength. We convert with `C = 1 / (λ N)` so the plots can be labelled in `λ`.

In [ ]:
from sklearn.linear_model import LogisticRegression

def fit_logreg(Xtr, ytr, penalty, lam):
    # TODO: when penalty is None, return LogisticRegression(penalty=None, max_iter=5000)
    # TODO: otherwise convert lambda to C = 1 / (lam * len(ytr)) and pass
    #       penalty=penalty, C=C, solver="liblinear"
    pass

lam = 0.02
models = {
    "none": fit_logreg(Xtr, ytr, None, lam),
    "L2":   fit_logreg(Xtr, ytr, "l2", lam),
    "L1":   fit_logreg(Xtr, ytr, "l1", lam),
}

# TODO: for each model print the weight norm, the number of EXACT zeros
#       (np.abs(coef) < 1e-8), and the validation accuracy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.0), sharey=True)
for ax, (name, m) in zip(axes, models.items()):
    coef = m.coef_.ravel()
    colours = ["#0D2137"] * 4 + ["#94A3B8"] * (len(coef) - 4)
    ax.bar(range(len(coef)), coef, color=colours)
    ax.axhline(0, color="#6B7280", lw=0.7)
    ax.set_title(f"{name}  (zeros: {int(np.sum(np.abs(coef) < 1e-8))})", fontsize=10)
    ax.set_xlabel("feature index")
axes[0].set_ylabel("weight")
fig.suptitle("Dark = the 4 real features, grey = the 16 noise features", fontsize=9)
plt.tight_layout(); plt.show()

Here L2 kept every feature and quietened the noise ones, while L1 drove most of
them to exactly zero. Check that the surviving L1 weights are concentrated on
the first four features — the ones that actually generated the data.

Read this as the typical pattern rather than a guarantee: ridge usually returns
a dense solution, and lasso *can* produce exact zeros but is not promised to
remove every weak feature at a given `λ`.

In [ ]:
# Self-check for Part F.
_l1 = models["L1"].coef_.ravel()
_l2 = models["L2"].coef_.ravel()
assert np.sum(np.abs(_l1) < 1e-8) > 0, \
    "L1 should zero out at least one weight - is solver='liblinear' and penalty='l1' set?"
assert np.sum(np.abs(_l2) < 1e-8) == 0, \
    "L2 should not produce exact zeros here"
assert np.linalg.norm(_l2) < np.linalg.norm(models["none"].coef_.ravel()), \
    "the L2 model should have a smaller weight norm than the unpenalised one"
print(f"Part F self-check passed. L1 zeros: {int(np.sum(np.abs(_l1) < 1e-8))}/{len(_l1)}")

## 10. Part G — The validation curve

`λ` is a hyperparameter: the optimiser cannot choose it. Sweep a **log** grid,
score each value on the validation set, and take the peak.

In [ ]:
lams = np.logspace(-4, 1, 24)
tr_acc, va_acc = [], []
for lam in lams:
    # TODO: fit an L2 model at this lambda
    # TODO: append its training accuracy and its validation accuracy
    pass
tr_acc, va_acc = np.array(tr_acc), np.array(va_acc)

# TODO: best = index of the highest validation accuracy  [np.argmax]
# TODO: print the best lambda and both accuracies there
# TODO: plot both curves against lambda with plt.semilogx and mark the best

In this run the training curve trends downward as `λ` rises. Be careful about
how far you generalise that.

- A stronger penalty can only **restrict** the fit, so the training *objective*
  being minimised cannot improve as `λ` grows.
- Training **accuracy** is a different quantity — a count of right and wrong
  answers — so it can sit flat, or move up and down by a case or two, without
  contradicting anything.

Either way, training performance is measured on data the model has already
seen, so it is not independent evidence about new data. That, and not
monotonicity, is why it cannot choose `λ`.

Validation accuracy rises, peaks, and then falls when the penalty starts
crushing real signal along with the noise. A wide gap between the two curves is
a symptom worth investigating, not by itself a measurement of overfitting.

In [ ]:
# Self-check for Part G. If any assertion fires, re-read the TODOs above.
# Note what is NOT asserted: training accuracy is not required to fall
# monotonically in lambda. It is a 0/1 count and may plateau or wobble.
assert len(tr_acc) == len(lams) and len(va_acc) == len(lams), \
    "tr_acc/va_acc must have one entry per lambda - did the loop body run?"
assert va_acc.max() <= 1.0 and va_acc.min() >= 0.0, "accuracies must lie in [0,1]"
assert tr_acc.max() <= 1.0 and tr_acc.min() >= 0.0, "accuracies must lie in [0,1]"
assert np.argmax(va_acc) not in (0, len(lams) - 1), \
    "the validation peak should be interior to the grid - widen the lambda range"
print("Part G self-check passed. best lambda =", round(lams[int(np.argmax(va_acc))], 5))
print("training accuracy first vs last:",
      round(tr_acc[0], 3), "->", round(tr_acc[-1], 3),
      "(a downward trend here; not a law)")

## 11. Part H — Cross-validation, and one honest test score

A single validation split is a lottery when data is scarce. 5-fold
cross-validation scores each `λ` five times and averages. Then, following step 6
of the lecture recipe, we refit on **training + validation combined** — the
validation split has done its job once `λ` is chosen, so holding it out any
longer just wastes data — and touch the test set **once**.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_mean = []
for lam in lams:
    C = 1.0 / (lam * len(ytr))
    m = LogisticRegression(penalty="l2", C=C, solver="liblinear", max_iter=5000)
    cv_mean.append(cross_val_score(m, Xtr, ytr, cv=5).mean())
cv_mean = np.array(cv_mean)

best_cv = int(np.argmax(cv_mean))
print(f"single-split best lambda    = {lams[best]:.4g}")
print(f"cross-validated best lambda = {lams[best_cv]:.4g}  "
      f"(mean CV accuracy {cv_mean[best_cv]:.3f})")

# Recipe step 6: lambda is settled, so refit on train + validation combined.
X_full = np.vstack([Xtr, Xva])
y_full = np.r_[ytr, yva]
final = fit_logreg(X_full, y_full, "l2", lams[best_cv])
print(f"refit on {len(y_full)} rows (train {len(ytr)} + val {len(yva)})")
print(f"TEST accuracy (reported once) = {final.score(Xte, yte):.3f}")

The two searches need not pick the identical `λ`; the cross-validated choice is
the more trustworthy one because it does not depend on a single lucky split.
Note that the test set appears exactly once in this entire notebook, on the last
line.

## 12. Part I — Why you must standardise first

A penalty charges by the size of the *weight*. Change a feature's units and its
weight changes, so the penalty charges a different amount for the same
information. Below, one feature is rescaled by a factor of 60 — nothing else
changes — and the fitted model moves.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_units = Xtr.copy()
X_units[:, 0] = X_units[:, 0] * 60.0      # same feature, "minutes" instead of "hours"

lam = 0.02
m_orig  = fit_logreg(Xtr,     ytr, "l2", lam)
m_units = fit_logreg(X_units, ytr, "l2", lam)
print("weight on feature 0, original units :", round(m_orig.coef_.ravel()[0], 4))
print("weight on feature 0, rescaled units :", round(m_units.coef_.ravel()[0], 5))
print("validation accuracy, original :", round(m_orig.score(Xva, yva), 3))

# Now standardise first, fitting the scaler on TRAINING data only.
scaler = StandardScaler().fit(X_units)
m_scaled = fit_logreg(scaler.transform(X_units), ytr, "l2", lam)
print("weight on feature 0, standardised   :", round(m_scaled.coef_.ravel()[0], 4))

In [ ]:
w_orig  = m_orig.coef_.ravel()[0]
w_units = m_units.coef_.ravel()[0]
print(f"weight ratio           = {w_orig / w_units:.1f}x smaller")
print(f"L2 charge, original    = {w_orig**2:.4f}")
print(f"L2 charge, rescaled    = {w_units**2:.6f}")
print(f"the penalty on this feature is {w_orig**2 / w_units**2:.0f}x cheaper "
      f"after the unit change")

Without a penalty the rescaled weight would be exactly 60 times smaller. With
the penalty in play it lands nearby, and the L2 charge on that feature — which
goes as the *square* of the weight — collapses by roughly three orders of
magnitude. The model was punished for a clerical choice about units, not for
anything about the students.

Standardising removes the problem. Fit the scaler on training data only; fitting
it on the full dataset leaks test information into training.

## 13. Part J — Responsible-use reflection

Write two or three sentences answering the following (edit the string below):

1. Your L1 model dropped the feature `topic match`. Why is it **unsafe** to
   conclude that topic match does not matter for learning?
2. A teacher asks: "if we gave this student extra practice, would they do
   better?" Can the model you just fitted answer that? Say why or why not.

In [ ]:
reflection = '''
TODO (student): write 2-3 sentences for each question.

1. A zero weight only means the feature added no predictive value beyond the
   features that were kept, at this lambda, on this sample; ...

2. The teacher's question is about an intervention, not a prediction, because ...
'''
print(reflection)

> **Check yourself.** A complete answer to (1) says *redundant, not irrelevant*
> and mentions that correlated features make L1's choice unstable. A complete
> answer to (2) says *no*, and explains that observing an association is not the
> same as predicting the effect of an intervention.

## 14. Summary

- The margin is `y(wᵀx + b)`; the corridor is `2/‖w‖`, so a wider margin means **smaller** weights.
- A support vector is a point with a non-zero dual coefficient `α`. Points on the margin are the *candidates*; in a degenerate case some of them are redundant and carry `α = 0`. With a soft margin, a support vector can also sit inside the corridor or violate it.
- The soft-margin SVM is hinge loss plus an L2 penalty; `C` is the price of a violation, and larger `C` means *less* regularization. The direction is universal; the exact `C`-to-`λ` constant depends on the objective — this lab uses the mean hinge loss, so `C = 1/(λN)`.
- L2 typically shrinks the weights and leaves a dense model; L1 **can** drive weak weights to exactly zero, which is where the readable short feature list comes from.
- Do not choose `λ` on training data (not independent evidence about new data) or on the test set (that would be tuning on the exam): use a validation curve or cross-validation, and report the test score once.
- Put features on a comparable scale before regularising, using training statistics only.
- A short feature list is a statement about prediction, not about cause.

**Submit** your completed notebook (all cells run, outputs visible) as part of
the lab portfolio.